# Summary Report : Mixed-Signal Analyzer (Finance & NLP)

## 1. Introduction and Background

The main objective of this project is to design and deploy an end-to-end Machine Learning pipeline capable of predicting the short-term trend of a financial asset. Specifically, the system must determine, through binary classification, whether the price of the target stock (in this case, Apple APPL) will rise or fall over a 5-day horizon.

The uniqueness and added value of this project lie in the integration of data. To best mimic a trader’s analysis, the model combines two distinct sources of information :
* Quantitative signals : Traditional time-series analysis, including price history and the calculation of technical indicators.
* Qualitative signals : Natural Language Processing (NLP) applied daily to financial news to capture market psychology and “sentiment.” 

Beyond pure prediction, the architecture of this project was designed to demonstrate complete mastery of the data value chain, including :
* Data Engineering : Creation of an automated ETL pipeline (Extraction via API, Transformation, and Loading).
* Statistical Modeling : Comparing advanced Machine Learning algorithms to solve a complex classification problem.
* Software Development : Structuring code into modular Python scripts, database management (SQL), and applying best practices for version control (Git).

## 2. Feature Engineering and Signal Justification

A model’s performance depends heavily on the quality of the explanatory variables it is fed. For this pipeline, we designed a hybrid dataset that captures both the intrinsic price dynamics (technical analysis) and exogenous market sentiment (sentiment analysis).

### 2.1. Quantitative Signals : Price Dynamics and Risk

* Moving Averages (SMA_20 and EMA_20) : The 20-day simple moving average (SMA) provides a baseline for the short- to medium-term trend. The exponential moving average (EMA), by placing greater weight on recent prices, allows the model to detect trend breaks more quickly.
* Historical Volatility (14 days) : This indicator measures market uncertainty. It is calculated using the moving standard deviation of daily returns, annualized according to the formula: $Volatility = \sigma_{14} \times \sqrt{252}$. Volatility spikes often precede major market reversals, a crucial signal for decision trees.
* Relative Strength Index (RSI_14) : This is a momentum oscillator that measures the speed of price movements to identify overbought or oversold conditions. Its mathematical formula is: $RSI = 100 - \frac{100}{1 + RS}$, where $RS$ (Relative Strength) represents the ratio of the average gains to the average losses over 14 days.

### 2.2. Qualitative Signals : NLP and Market Psychology

Pure quantitative models suffer from an inherent lag because they respond only to past prices. The integration of textual data aims to give the model the ability to anticipate based on the flow of information.

* The Choice of FinBERT : General-purpose NLP models often fail to capture the nuances of financial jargon (for example, the word “drop” can be positive when referring to the unemployment rate). We implemented FinBERT, a Transformer-based model specifically retrained by ProsusAI on a massive financial corpus (Financial PhraseBank).
* Daily Aggregation (daily_sentiment and new_volumes) : FinBERT’s raw predictions (probabilities for the Positive, Negative, and Neutral classes) were weighted and aggregated on a daily basis. This allows us to transform an unstructured news feed into a continuous time series that aligns perfectly with our market data in the SQL table `fact_news_sentiment`.

## 3. Évaluation et Performances des Modèles

### 3.1. Méthodologie d'Évaluation et Validation Temporelle

En modélisation financière, l'utilisation d'une validation croisée classique (type K-Fold aléatoire) est à proscrire, car elle entraînerait une fuite de données (data leakage) du futur vers le passé. Pour garantir l'intégrité de nos tests sur l'action Apple, nous avons mis en place plusieurs outils : 
* Un découpage chronologique strict (80% / 20%) : Le modèle s'entraîne uniquement sur le passé lointain et est évalué exclusivement sur les données les plus récentes.  
* Un TimeSeriesSplit (Cross-Validation temporelle) : L'optimisation des hyperparamètres via GridSearchCV a respecté l'ordre chronologique des blocs d'entraînement pour simuler un véritable environnement de trading en conditions réelles.  
* Une formulation en classification binaire : La variable cible indique si le cours de clôture à un horizon de 5 jours sera supérieur ($1$) ou inférieur ($0$) au cours actuel.  

### 3.2. Analyse Comparative : Random Forest vs XGBoost

Les deux algorithmes ensemblistes testés ont révélé des dynamiques d'apprentissage radicalement opposées sur notre jeu de données.

* Le comportement du Random Forest (Modèle Conservateur) : Le Random Forest construit des arbres indépendants en parallèle. Face au bruit inhérent des séries temporelles boursières, la moyenne des votes de la forêt a adopté une stratégie de lissage extrême. Sur l'échantillon de test, le modèle a privilégié la classe majoritaire, démontrant une incapacité à isoler les signaux faibles de retournement de tendance. Bien qu'il cherche à contrer le surapprentissage, sa structure indépendante s'est montrée trop rigide pour capter la non-linéarité des signaux mixtes (prix + sentiment).
* Le comportement de XGBoost (Gradient Boosting Séquentiel) : À l'inverse, XGBoost construit ses arbres de manière itérative, chaque nouvel arbre est spécifiquement entraîné pour corriger les erreurs résiduelles des arbres précédents. Cette approche séquentielle lui a permis de s'adapter finement aux variations de prix et de surperformer, atteignant une précision globale supérieure à celle de Random Forest.

### 3.3.  Analyse des Métriques Avancées (ROC, Precision-Recall et Confusion)

Pour valider la robustesse de XGBoost, l'analyse ne se limite pas à l'exactitude globale (Accuracy) :

* La Matrice de Confusion : Elle met en évidence la capacité du modèle à identifier correctement les zones de Hausse et de Baisse, minimisant les faux signaux par rapport au Random Forest.
* Les Courbes ROC et Precision-Recall : L'analyse des scores de probabilité à travers l'aire sous la courbe (AUC) confirme la robustesse du classifieur, lui conférant une capacité discriminante satisfaisante pour envisager une automatisation des ordres de décision.

### Comparatif : Random Forest vs XGBoost

* **Random Forest :** Ce modèle ensembliste s'est montré trop "conservateur" face au bruit des données synthétiques, finissant par prédire une classe unique (lissant les signaux). 
* **XGBoost :** En construisant ses arbres de manière séquentielle pour corriger ses propres erreurs (Gradient Boosting), XGBoost a réussi à capter les patterns sous-jacents, obtenant une Accuracy supérieure à 70%.

*(Insérez ici une cellule de code pour afficher vos graphiques : Matrice de Confusion XGBoost et Courbes ROC/PR)*

## 4. Importance des Variables et Conclusion Finale

*(Insérez ici une cellule de code pour afficher le graphique Feature Importance de XGBoost)*

**Le sentiment des actualités permet-il réellement d'améliorer les prédictions ?**
L'analyse de l'importance des variables (Feature Importance) montre que... *(à vous de compléter en lisant votre graphique : si daily_sentiment est bien classé, le NLP a apporté de la valeur !)*.

En conclusion, ce projet valide la faisabilité technique d'un pipeline hybride automatisé, tout en soulignant qu'un passage en production réel nécessiterait un historique de données (NLP et Prix) sur plusieurs années pour maximiser la robustesse de l'algorithme.